# Evaluación RAG — Fase 4: Embeddings y Espacio Latente

Mide la calidad de la representación vectorial de tus nodos en Pinecone, de forma aislada del resto del pipeline RAG.

Métricas implementadas:
- **Coherencia Semántica** (similitud coseno intra-cluster) + baseline de control
- **Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index** — sobre el vector original, nunca sobre la proyección 2D/3D
- **kNN Label Purity** (k=5, k=10) — coherente con los mismos K de la Fase 1
- **PCA / t-SNE / UMAP** + **Trustworthiness** — solo para inspección visual
- **Caso de prueba especial**: los 2 periodos que cruzan sector

Todo se evalúa frente a tres variables de agrupación: `periodo`, `indice_sector` y `regimen_mercado`.

### Documentos necesarios

1. `embeddings_metadata.csv` (adjunto) — metadatos de los 9082 nodos (`node_id`, `periodo`, `indice_sector`, `regimen_mercado`, `ticker`, `fecha`), ya generado desde tu `noticias_nodes.json` real.
2. **Los vectores de embedding reales** — aún pendiente de decidir su origen (quedó abierto en la Fase 4 del diseño). Este notebook soporta las dos opciones; configura `VECTOR_SOURCE` más abajo:
   - `"pinecone"`: se conecta a tu índice y hace `fetch()` de los vectores por `node_id` (requiere `PINECONE_API_KEY` e `PINECONE_INDEX_NAME`).
   - `"local_file"`: subes tú un archivo `embeddings_vectors.json` con formato `[{"node_id": "...", "vector": [0.1, 0.2, ...]}, ...]` (por ejemplo, exportado directamente de tu pipeline de indexación antes de subir a Pinecone).

> ⚠️ Por volumen (9082 nodos × dimensión del embedding), si usas `"pinecone"` el fetch puede tardar varios minutos y debe hacerse en lotes (ya implementado abajo).

In [ ]:
# @title 1. Instalación de dependencias
!pip install -q pandas numpy scikit-learn matplotlib seaborn umap-learn pinecone

## 2. Subir archivos

Sube `embeddings_metadata.csv`, y si usas `VECTOR_SOURCE = "local_file"`, también `embeddings_vectors.json`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Archivos subidos:", list(uploaded.keys()))

## 3. Configuración

In [ ]:
# @title Configuración
VECTOR_SOURCE = "pinecone"  # @param ["pinecone", "local_file"]

METADATA_PATH = "embeddings_metadata.csv"  # @param {type:"string"}
LOCAL_VECTORS_PATH = "embeddings_vectors.json"  # @param {type:"string"}

try:
    from google.colab import userdata
    PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
except ImportError:
    import getpass
    PINECONE_API_KEY = getpass.getpass("PINECONE_API_KEY: ")
PINECONE_INDEX_NAME = "noticias-financieras"  # @param {type:"string"}
PINECONE_NAMESPACE = ""  # @param {type:"string"}
FETCH_BATCH_SIZE = 100  # @param {type:"integer"}

K_VALUES = [5, 10]
GROUP_VARS = ["periodo", "indice_sector", "regimen_mercado"]


## 4. Cargar metadatos y vectores

Si usas `VECTOR_SOURCE = "pinecone"`, el notebook no depende de los `node_id` de
`embeddings_metadata.csv` para el `fetch()`. Esos IDs vienen de una ejecución concreta del
pipeline de segmentación de LlamaIndex y son aleatorios en cada ejecución — si el índice se
(re)indexó en otro momento, esos IDs no existen de verdad en Pinecone (el fetch no da error,
simplemente no encuentra nada).

En su lugar, se le pregunta directamente a Pinecone qué IDs existen ahora mismo (`index.list()`),
y se trae el vector Y los metadatos de cada uno directamente del índice (`fetch()` ya incluye
`metadata`), sin depender del CSV local para el emparejamiento. El CSV local solo se usa como
referencia informativa del tamaño esperado del corpus.

In [ ]:
import pandas as pd
import numpy as np
import json

meta = pd.read_csv(METADATA_PATH)
print(f"Nodos en metadatos locales: {len(meta)}")

vectors_by_id = {}

if VECTOR_SOURCE == "local_file":
    with open(LOCAL_VECTORS_PATH, encoding="utf-8") as f:
        raw = json.load(f)
    for item in raw:
        vectors_by_id[item["node_id"]] = np.array(item["vector"], dtype=np.float32)
    print(f"Vectores cargados desde archivo local: {len(vectors_by_id)}")

    meta_with_vectors = meta[meta["node_id"].isin(vectors_by_id.keys())].reset_index(drop=True)
    faltantes = len(meta) - len(meta_with_vectors)
    if faltantes > 0:
        print(f"\u26a0\ufe0f {faltantes} nodos sin vector encontrado (excluidos del an\u00e1lisis).")

elif VECTOR_SOURCE == "pinecone":
    # NO usamos los node_id del CSV local para el fetch: los node_id de LlamaIndex son
    # aleatorios en cada ejecuci\u00f3n del pipeline de indexaci\u00f3n, as\u00ed que un
    # embeddings_metadata.csv generado a partir de una ejecuci\u00f3n concreta NUNCA coincidir\u00e1
    # con los IDs reales si el \u00edndice se (re)index\u00f3 en otro momento (fetch por ID
    # simplemente no encuentra nada, sin dar error).
    #
    # En su lugar: preguntamos a Pinecone qu\u00e9 IDs existen REALMENTE ahora mismo (index.list()),
    # y traemos sus vectores Y sus metadatos directamente del \u00edndice (fetch ya incluye
    # metadata), sin depender del CSV local para nada m\u00e1s que informar del tama\u00f1o esperado.
    from pinecone import Pinecone
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index = pc.Index(PINECONE_INDEX_NAME)

    print("Enumerando IDs reales existentes en el \u00edndice de Pinecone...")
    real_ids = []
    list_kwargs = {}
    if PINECONE_NAMESPACE:
        list_kwargs["namespace"] = PINECONE_NAMESPACE
    for page in index.list(**list_kwargs):
        real_ids.extend([v.id for v in page.vectors])
    print(f"IDs reales encontrados en el \u00edndice: {len(real_ids)}  (metadatos locales tienen: {len(meta)})")

    if len(real_ids) == 0:
        raise RuntimeError(
            "index.list() no devolvi\u00f3 ning\u00fan ID. Revisa PINECONE_INDEX_NAME y "
            "PINECONE_NAMESPACE (si tu \u00edndice usa namespaces, aseg\u00farate de indicarlo)."
        )

    registros_metadata = []
    for i in range(0, len(real_ids), FETCH_BATCH_SIZE):
        batch_ids = real_ids[i:i + FETCH_BATCH_SIZE]
        fetch_kwargs = dict(ids=batch_ids)
        if PINECONE_NAMESPACE:
            fetch_kwargs["namespace"] = PINECONE_NAMESPACE
        result = index.fetch(**fetch_kwargs)
        vectors_dict = result.vectors if hasattr(result, "vectors") else result["vectors"]
        for node_id, record in vectors_dict.items():
            values = record.values if hasattr(record, "values") else record["values"]
            md_ = record.metadata if hasattr(record, "metadata") else record.get("metadata", {})
            md_ = md_ or {}
            vectors_by_id[node_id] = np.array(values, dtype=np.float32)
            registros_metadata.append({
                "node_id": node_id,
                "url": md_.get("url"),
                "ticker": md_.get("ticker"),
                "indice_sector": md_.get("indice_sector"),
                "periodo": md_.get("periodo"),
                "regimen_mercado": md_.get("regimen_mercado"),
                "fecha": (md_.get("fecha") or "")[:10],
            })
        if (i // FETCH_BATCH_SIZE) % 10 == 0:
            print(f"  {min(i + FETCH_BATCH_SIZE, len(real_ids))}/{len(real_ids)} vectores obtenidos...")

    print(f"Vectores obtenidos de Pinecone: {len(vectors_by_id)}")
    meta_with_vectors = pd.DataFrame(registros_metadata)

    faltan_metadata = meta_with_vectors[["periodo", "indice_sector", "regimen_mercado"]].isna().any(axis=1).sum()
    if faltan_metadata > 0:
        print(f"\u26a0\ufe0f {faltan_metadata} vectores sin metadata completa en Pinecone (revisa los nombres de campo "
              f"si tu pipeline de indexaci\u00f3n us\u00f3 claves distintas a periodo/indice_sector/regimen_mercado).")

else:
    raise ValueError("VECTOR_SOURCE debe ser \'pinecone\' o \'local_file\'")

X = np.stack([vectors_by_id[nid] for nid in meta_with_vectors["node_id"]])
print(f"Matriz de vectores: {X.shape}")

## 5. Coherencia Semántica + baseline de control

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def mean_intra_group_similarity(X, labels, group_value, sample_max=300):
    idx = np.where(labels == group_value)[0]
    if len(idx) < 2:
        return np.nan
    if len(idx) > sample_max:
        idx = np.random.choice(idx, sample_max, replace=False)
    sims = cosine_similarity(X[idx])
    iu = np.triu_indices_from(sims, k=1)
    return sims[iu].mean()

def baseline_similarity(X, sample_max=500):
    idx = np.random.choice(len(X), min(sample_max, len(X)), replace=False)
    sims = cosine_similarity(X[idx])
    iu = np.triu_indices_from(sims, k=1)
    return sims[iu].mean()

baseline = baseline_similarity(X)
print(f"Similitud media del corpus completo (baseline): {baseline:.4f}\n")

coherencia_rows = []
for group_var in GROUP_VARS:
    labels = meta_with_vectors[group_var].values
    for value in pd.unique(labels):
        coh = mean_intra_group_similarity(X, labels, value)
        coherencia_rows.append({
            "variable_agrupacion": group_var, "grupo": value,
            "coherencia": coh, "coherencia_relativa": coh - baseline,
            "n_nodos": int((labels == value).sum()),
        })

coherencia_df = pd.DataFrame(coherencia_rows).sort_values(["variable_agrupacion", "coherencia_relativa"], ascending=[True, False])
coherencia_df

## 6. Silhouette, Davies-Bouldin y Calinski-Harabasz (sobre el vector original)

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

cluster_metric_rows = []
for group_var in GROUP_VARS:
    labels = meta_with_vectors[group_var].values
    if len(pd.unique(labels)) < 2:
        continue
    sil = silhouette_score(X, labels, metric="cosine")
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    cluster_metric_rows.append({
        "variable_agrupacion": group_var,
        "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch,
        "n_grupos": len(pd.unique(labels)),
    })

cluster_metrics_df = pd.DataFrame(cluster_metric_rows)
cluster_metrics_df

## 7. Pureza de Vecinos Cercanos (kNN Label Agreement)

In [ ]:
from sklearn.neighbors import NearestNeighbors

max_k = max(K_VALUES)
nn_model = NearestNeighbors(n_neighbors=max_k + 1, metric="cosine").fit(X)
_, neighbor_idx = nn_model.kneighbors(X)
neighbor_idx = neighbor_idx[:, 1:]  # excluir el propio punto (vecino 0 = si mismo)

knn_rows = []
for group_var in GROUP_VARS:
    labels = meta_with_vectors[group_var].values
    for k in K_VALUES:
        purities = []
        for i in range(len(X)):
            own_label = labels[i]
            neighbor_labels = labels[neighbor_idx[i, :k]]
            purity = (neighbor_labels == own_label).mean()
            purities.append(purity)
        knn_rows.append({
            "variable_agrupacion": group_var, "k": k,
            "purity_media": np.mean(purities), "purity_mediana": np.median(purities),
        })

knn_df = pd.DataFrame(knn_rows)
knn_df

## 8. Reducción de dimensionalidad (visual) + Trustworthiness

⚠️ Estas proyecciones son solo para inspección visual — las métricas cuantitativas ya se calcularon sobre el vector original en las secciones anteriores.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
import umap

# submuestreo si el corpus es muy grande, para que t-SNE/UMAP sean rápidos
sample_size = min(3000, len(X))
sample_idx = np.random.choice(len(X), sample_size, replace=False)
X_sample = X[sample_idx]
meta_sample = meta_with_vectors.iloc[sample_idx].reset_index(drop=True)

pca_2d = PCA(n_components=2, random_state=42).fit_transform(X_sample)
tsne_2d = TSNE(n_components=2, random_state=42, init="pca", perplexity=30).fit_transform(X_sample)
umap_2d = umap.UMAP(n_components=2, random_state=42).fit_transform(X_sample)

trust_pca = trustworthiness(X_sample, pca_2d, n_neighbors=10)
trust_tsne = trustworthiness(X_sample, tsne_2d, n_neighbors=10)
trust_umap = trustworthiness(X_sample, umap_2d, n_neighbors=10)

print(f"Trustworthiness PCA:  {trust_pca:.4f}")
print(f"Trustworthiness t-SNE: {trust_tsne:.4f}")
print(f"Trustworthiness UMAP:  {trust_umap:.4f}")
print("(< ~0.85 sugiere que la proyección puede no reflejar fielmente la geometría real del embedding)")

In [ ]:
import matplotlib.pyplot as plt

COLOR_MAP = {"bearish": "#B5474D", "bullish": "#3E8E58", "sideways": "#C7A24D"}
colors = meta_sample["regimen_mercado"].map(COLOR_MAP)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (proj, name, trust) in zip(
    axes,
    [(pca_2d, "PCA", trust_pca), (tsne_2d, "t-SNE", trust_tsne), (umap_2d, "UMAP", trust_umap)],
):
    ax.scatter(proj[:, 0], proj[:, 1], c=colors, s=6, alpha=0.6)
    ax.set_title(f"{name} (Trustworthiness={trust:.3f})")
    ax.set_xticks([]); ax.set_yticks([])

handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, label=l, markersize=8)
           for l, c in COLOR_MAP.items()]
fig.legend(handles=handles, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.05))
plt.tight_layout()
plt.savefig("proyeccion_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Caso de prueba especial: periodos que cruzan sector

Los 2 periodos que aparecen en dos sectores distintos (`"caidas crisis financiera"` en S&P 500 y Sector Defensa; `"subidas vacunas reapertura"` en Euro Stoxx 50 y Sector Turismo/Aerolíneas). Comprobamos si el embedding los separa por sector (bueno) o los colapsa juntos (el embedding solo capta el tono, no el sector).

In [ ]:
CROSS_SECTOR_PERIODS = ["caidas crisis financiera", "subidas vacunas reapertura"]

for periodo in CROSS_SECTOR_PERIODS:
    subset = meta_with_vectors[meta_with_vectors["periodo"] == periodo]
    sectores = subset["indice_sector"].unique()
    if len(sectores) < 2:
        print(f"'{periodo}': no se encontraron ambos sectores en los nodos con vector disponible, se omite.")
        continue

    idx = subset.index.values
    X_sub = X[idx]
    labels_sector = subset["indice_sector"].values

    sil = silhouette_score(X_sub, labels_sector, metric="cosine") if len(np.unique(labels_sector)) > 1 else np.nan
    print(f"Periodo: '{periodo}'  (sectores: {list(sectores)}, n={len(subset)})")
    print(f"  Silhouette por sector dentro de este periodo: {sil:.4f}")
    print(f"  (cercano a 1 = el embedding SÍ distingue el sector incluso con el mismo tono/periodo;")
    print(f"   cercano a 0 o negativo = el embedding colapsa ambos sectores juntos)\n")

## 10. Guardar resultados

In [ ]:
with pd.ExcelWriter("resultados_embeddings.xlsx") as writer:
    coherencia_df.to_excel(writer, sheet_name="coherencia_semantica", index=False)
    cluster_metrics_df.to_excel(writer, sheet_name="silhouette_db_ch", index=False)
    knn_df.to_excel(writer, sheet_name="knn_purity", index=False)

from google.colab import files as colab_files
colab_files.download("resultados_embeddings.xlsx")
colab_files.download("proyeccion_embeddings.png")

print("Guardado resultados_embeddings.xlsx con 3 hojas.")